**SECTION 1 - TEST RUN**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE='/content/drive/MyDrive/mtb_drug_targets/'
!apt-get install -y ncbi-blast+ -q
print("BLAST installed!")
!blastp -version

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading package lists...
Building dependency tree...
Reading state information...
ncbi-blast+ is already the newest version (2.12.0+ds-3build1).
0 upgraded, 0 newly installed, 0 to remove and 29 not upgraded.
BLAST installed!
blastp: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


In [ ]:
#loading of MTB data
import pandas as pd
df_mtb = pd.read_csv(BASE + 'results/proteome_final.csv')
print(f"MTB proteins loaded: {len(df_mtb)}")
print(df_mtb.head(3))

MTB proteins loaded: 3980
                 Protein_ID                                 protein_name  \
0  sp|A0A089QRB9|MSL3_MYCTU                       Mycolipanoate synthase   
1      sp|I6WXK4|PTPB_MYCTU  Triple specificity protein phosphatase PtpB   
2     sp|I6WZG6|ENCAP_MYCTU              Type 1 encapsulin shell protein   

   Length                                   Protein Sequence  
0    2085  MRTATATSVAVIGMACRLPGGIDSPQRLWEALLRGDDLVGEIPADR...  
1     276  MAVRELPGAWNFRDVADTATALRPGRLFRSSELSRLDDAGRATLRR...  
2     265  MNNLYRDLAPVTEAAWAEIELEAARTFKRHIAGRRVVDVSDPGGPV...  


In [ ]:
#Downloading of human proteome
!wget -q "https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=proteome:UP000005640" -O /content/human_proteome.fasta
print("Download completed!")
!pip install biopython -q
from  Bio import SeqIO
fasta_path="/content/human_proteome.fasta"
with open(fasta_path, 'rt') as fasta_handle:
  sequences=list(SeqIO.parse(fasta_handle, 'fasta'))
print(f"Total number of proteins: {len(sequences)}")

Download completed!
Total number of proteins: 147506


In [ ]:
!apt-get update -q
!apt-get install -y ncbi-blast+ -q
!makeblastdb -version

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 MB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,489 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,357 kB]
Get:14 http://archive.ubu

In [ ]:
#Building of BLAST database
!makeblastdb -in /content/human_proteome.fasta \
             -dbtype prot \
             -out /content/human_blast_db \
             -title "Human Proteome"
print("BLAST database ready!")



Building a new DB, current time: 06/11/2026 11:49:19
New DB name:   /content/human_blast_db
New DB title:  Human Proteome
Sequence type: Protein
Deleted existing Protein BLAST database named /content/human_blast_db
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 147506 sequences in 6.20778 seconds.


BLAST database ready!


In [ ]:
#Run BLAST
import pandas as pd
from Bio import SeqIO
df_mtb = pd.read_csv(BASE + 'results/proteome_final.csv')
df_test = df_mtb.head(10)
with open('/content/test_mtb.fasta','w') as f:
  for _, row in df_test.iterrows(): #doing this part since BLAST cannot read dataframe, so just converting them back to a FASTA file
   f.write(f">{row['Protein_ID']}\n{row['Protein Sequence']}\n")
print(f"Test FASTA created with {len(df_test)} proteins")

Test FASTA created with 10 proteins


In [ ]:
!blastp -query /content/test_mtb.fasta \
        -db /content/human_blast_db \
        -out /content/blast_test_results.txt \
        -outfmt 6 \
        -evalue 1 \
        -num_threads 2
print("BLAST complete!")
with open('/content/blast_test_results.txt', 'r') as f:
    content = f.read()
print(content)

BLAST complete!
sp|A0A089QRB9|MSL3_MYCTU	tr|A0ACI8U4L2|A0ACI8U4L2_HUMAN	27.675	1084	674	38	9	1060	4	1009	1.57e-75	283
sp|A0A089QRB9|MSL3_MYCTU	tr|A0ACI8U4L2|A0ACI8U4L2_HUMAN	32.364	550	322	14	1407	1925	1537	2067	5.60e-53	209
sp|A0A089QRB9|MSL3_MYCTU	tr|A0A0U1RQF0|A0A0U1RQF0_HUMAN	27.675	1084	674	38	9	1060	4	1009	2.15e-75	282
sp|A0A089QRB9|MSL3_MYCTU	tr|A0A0U1RQF0|A0A0U1RQF0_HUMAN	30.256	704	401	20	1407	2071	1535	2187	1.04e-53	211
sp|A0A089QRB9|MSL3_MYCTU	sp|P49327|FAS_HUMAN	27.675	1084	674	38	9	1060	4	1009	2.35e-75	282
sp|A0A089QRB9|MSL3_MYCTU	sp|P49327|FAS_HUMAN	30.256	704	401	20	1407	2071	1537	2189	1.04e-53	211
sp|A0A089QRB9|MSL3_MYCTU	tr|A0ACI8U5M9|A0ACI8U5M9_HUMAN	27.905	1093	658	38	9	1060	4	1007	2.97e-75	281
sp|A0A089QRB9|MSL3_MYCTU	tr|A0ACI8U5M9|A0ACI8U5M9_HUMAN	30.256	704	401	20	1407	2071	1535	2187	7.51e-54	211
sp|A0A089QRB9|MSL3_MYCTU	tr|A0ACI8U300|A0ACI8U300_HUMAN	27.482	1088	679	38	9	1060	4	1017	1.58e-74	279
sp|A0A089QRB9|MSL3_MYCTU	tr|A0ACI8U300|A0ACI8U300_HUMAN	30.256	704	4

In [ ]:
#Loading BLAST test results into Pandas
import pandas as pd
df_blast=pd.read_csv("/content/blast_test_results.txt", sep='\t', header = None,
                        names=['query', 'subject', 'pident', 'length',
            'mismatch', 'gapopen', 'qstart', 'qend',
            'sstart', 'send', 'evalue', 'bitscore'])
print(df_blast.head())
print(df_blast.shape)

                      query                         subject  pident  length  \
0  sp|A0A089QRB9|MSL3_MYCTU  tr|A0ACI8U4L2|A0ACI8U4L2_HUMAN  27.675    1084   
1  sp|A0A089QRB9|MSL3_MYCTU  tr|A0ACI8U4L2|A0ACI8U4L2_HUMAN  32.364     550   
2  sp|A0A089QRB9|MSL3_MYCTU  tr|A0A0U1RQF0|A0A0U1RQF0_HUMAN  27.675    1084   
3  sp|A0A089QRB9|MSL3_MYCTU  tr|A0A0U1RQF0|A0A0U1RQF0_HUMAN  30.256     704   
4  sp|A0A089QRB9|MSL3_MYCTU             sp|P49327|FAS_HUMAN  27.675    1084   

   mismatch  gapopen  qstart  qend  sstart  send        evalue  bitscore  
0       674       38       9  1060       4  1009  1.570000e-75     283.0  
1       322       14    1407  1925    1537  2067  5.600000e-53     209.0  
2       674       38       9  1060       4  1009  2.150000e-75     282.0  
3       401       20    1407  2071    1535  2187  1.040000e-53     211.0  
4       674       38       9  1060       4  1009  2.350000e-75     282.0  
(525, 12)


In [ ]:
#filtering of candidates
df_candidates=df_blast[df_blast['evalue'] > 0.001]
print("Total number of candidates:", len(df_candidates))
print(df_candidates[['query', 'subject', 'evalue']].head(10))

Total number of candidates: 123
                       query                         subject  evalue
77  sp|A0A089QRB9|MSL3_MYCTU  tr|A0ACI8S826|A0ACI8S826_HUMAN   0.002
78  sp|A0A089QRB9|MSL3_MYCTU          tr|C9K0F7|C9K0F7_HUMAN   0.002
79  sp|A0A089QRB9|MSL3_MYCTU  tr|A0ACI8ULH9|A0ACI8ULH9_HUMAN   0.002
80  sp|A0A089QRB9|MSL3_MYCTU  tr|A0ACI8UNC1|A0ACI8UNC1_HUMAN   0.002
81  sp|A0A089QRB9|MSL3_MYCTU          tr|K7ER81|K7ER81_HUMAN   0.003
82  sp|A0A089QRB9|MSL3_MYCTU  tr|A0ACI8VP07|A0ACI8VP07_HUMAN   0.009
83  sp|A0A089QRB9|MSL3_MYCTU          tr|C9JAL0|C9JAL0_HUMAN   0.013
84  sp|A0A089QRB9|MSL3_MYCTU          tr|K7ERT7|K7ERT7_HUMAN   0.014
85  sp|A0A089QRB9|MSL3_MYCTU          tr|G3V1R2|G3V1R2_HUMAN   0.037
86  sp|A0A089QRB9|MSL3_MYCTU            sp|P51659|DHB4_HUMAN   0.180


In [ ]:
#saving cleaner version
df_candidates.to_csv(BASE + 'results/blast_candidates_test.csv', index=False)
print("cleaner version saved!")

cleaner version saved!


**SECTION 2 - FULL RUN**

In [39]:
#setup
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mtb_drug_targets/'
!apt-get install -y ncbi-blast+ -q
!pip install biopython -q
!cp /content/drive/MyDrive/mtb_drug_targets/data/human_blast_db* /content/
print("Setup complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading package lists...
Building dependency tree...
Reading state information...
ncbi-blast+ is already the newest version (2.12.0+ds-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Setup complete!


In [40]:
#Loading MTB data
import pandas as pd
df_mtb = pd.read_csv(BASE + 'results/proteome_final.csv')
print(f"MTB proteins loaded: {len(df_mtb)}")
print(df_mtb.head(3))

MTB proteins loaded: 3980
                 Protein_ID                                 protein_name  \
0  sp|A0A089QRB9|MSL3_MYCTU                       Mycolipanoate synthase   
1      sp|I6WXK4|PTPB_MYCTU  Triple specificity protein phosphatase PtpB   
2     sp|I6WZG6|ENCAP_MYCTU              Type 1 encapsulin shell protein   

   Length                                   Protein Sequence  
0    2085  MRTATATSVAVIGMACRLPGGIDSPQRLWEALLRGDDLVGEIPADR...  
1     276  MAVRELPGAWNFRDVADTATALRPGRLFRSSELSRLDDAGRATLRR...  
2     265  MNNLYRDLAPVTEAAWAEIELEAARTFKRHIAGRRVVDVSDPGGPV...  


In [41]:
#conversion of all 3980 proteins to FASTA format for BLAST
with open(BASE + 'data/full_mtb.fasta', 'w') as f:
    for _, row in df_mtb.iterrows():
        f.write(f">{row['Protein_ID']}\n{row['Protein Sequence']}\n")
print(f"FASTA file created with {len(df_mtb)} proteins!")

FASTA file created with 3980 proteins!


In [42]:
#Running of full BLAST
!blastp -query {BASE}data/full_mtb.fasta \
        -db /content/human_blast_db \
        -out {BASE}results/blast_full_results.txt \
        -outfmt 6 \
        -evalue 1 \
        -num_threads 2
print("BLAST complete!")

BLAST complete!


In [43]:
#loading of BLAST results
import pandas as pd
df_blast = pd.read_csv(
    BASE + 'results/blast_full_results.txt',
    sep='\t',
    header=None,
    names=['query', 'subject', 'pident', 'length',
           'mismatch', 'gapopen', 'qstart', 'qend',
           'sstart', 'send', 'evalue', 'bitscore'])
print(f"Total BLAST hits: {len(df_blast)}")
print(df_blast.head(3))

Total BLAST hits: 37093
                      query                         subject  pident  length  \
0  sp|A0A089QRB9|MSL3_MYCTU  tr|A0A0U1RQF0|A0A0U1RQF0_HUMAN  27.675    1084   
1  sp|A0A089QRB9|MSL3_MYCTU  tr|A0A0U1RQF0|A0A0U1RQF0_HUMAN  30.256     704   
2  sp|A0A089QRB9|MSL3_MYCTU             sp|P49327|FAS_HUMAN  27.675    1084   

   mismatch  gapopen  qstart  qend  sstart  send        evalue  bitscore  
0       674       38       9  1060       4  1009  4.920000e-76     282.0  
1       401       20    1407  2071    1535  2187  2.390000e-54     211.0  
2       674       38       9  1060       4  1009  5.360000e-76     282.0  


In [44]:
#Filtering of Non-Homologous candidates
df_candidates = df_blast[df_blast['evalue'] > 0.001]
print(f"Total hits: {len(df_blast)}")
print(f"Non-homologous hits: {len(df_candidates)}")
print(f"Unique MTB candidate proteins: {df_candidates['query'].nunique()}")

Total hits: 37093
Non-homologous hits: 11819
Unique MTB candidate proteins: 2162


In [45]:
#saving of final candidates
df_unique_candidates = df_candidates.groupby('query').apply(
    lambda x: x.nsmallest(1, 'evalue')
).reset_index(drop=True)
print(f"Final candidates: {len(df_unique_candidates)} unique MTB proteins")
df_unique_candidates.to_csv(BASE + 'results/final_candidates.csv', index=False)
print("Final candidates saved!")

Final candidates: 2162 unique MTB proteins
Final candidates saved!


/tmp/ipykernel_2774/958361380.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_unique_candidates = df_candidates.groupby('query').apply(
